In [ ]:
# Run this cell if you're using from colab
#!git clone https://github.com/R-Oc-A/HackathonPastryLPV.git
#!wget https://github.com/R-Oc-A/HackathonPastryLPV/releases/download/IntensityGrids/grids.tar.gz
#!tar -xzf grids.tar.gz
#!pip install https://github.com/R-Oc-A/HackathonPastryLPV/releases/download/wheel/pastrypy-010-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
#!pip install tomli_w
#import sys
#sys.path.append('/content/HackathonPastryLPV')
#import os
#os.environ["GRIDS"]="/content/grids/ema_parquets/"

In [1]:
import pastrypy as psp
import pulsation_description as plsd
import line_profile_description as lpd
import tomli_w
import os
import polars as pl
import matplotlib.pyplot as plt

# Pulstar configuration
Here you specify the star you'll be modelling as well as the modes of pulsation

In [ ]:
#Taken from a Simbad quick Query and from Teltings paper
mode=plsd.Mode(l=2,m=1,
                rel_dr=0.0024,
                k=0.03,frequency=5.38,
                phase_offset=0.0,
                rel_dtemp=2.62,
                phase_rel_dtemp=180.0,
                rel_dg=10.0,
                phase_rel_dg=34.0,
                rotation_effects = "PerturbativeCoriolis")
star_data=plsd.StarData(mass=10.0,
                        radius=6.93,
                        effective_temperature=21642.0,
                        v_omega=20.0,
                        inclination_angle=45.0)
time_points=plsd.TimePoints(Uniform=plsd.UniformTime(start=0.0,end=0.0,step=0.01))
#mesh=plsd.Mesh(Sphere=plsd.SphericalStar(theta_step=2.0,phi_step=4.0))
mesh=plsd.Mesh(HSphere=plsd.HealpixStar(depth=3))

pulsconfig=plsd.PulstarConfig(mode_data=[mode],star_data=star_data,time_points=time_points,mesh=mesh)
pulsconfig_dict=pulsconfig.model_dump(exclude_none=True)
puls_toml_string=tomli_w.dumps(pulsconfig_dict)

[[mode_data]]
l = 2
m = 1
rel_dr = 0.0024
k = 0.03
frequency = 5.38
phase_offset = 0.0
rel_dtemp = 2.62
phase_rel_dtemp = 180.0
rel_dg = 10.0
phase_rel_dg = 34.0
rotation_effects = "PerturbativeCoriolis"

[star_data]
mass = 10.0
radius = 6.93
effective_temperature = 21642.0
v_omega = 20.0
inclination_angle = 45.0

[time_points.Uniform]
start = 0.0
end = 0.0
step = 0.01

[mesh.HSphere]
depth = 3



# Profile configuration
Here you specify the line profile variability you want to observe.

In [ ]:
#Taken from a Simbad quick Query
wl_range=lpd.WavelengthRange(start=4551.0,end=4555.0,step=0.0033)
#path_to_grids="../profile/grids/"
#path_to_grids = os.getenv("GRIDS")
#path_to_grids = f"{os.getenv("GRIDS")}ema_parquets/"
#print(path_to_grids)
grid1=lpd.IntensityGrid(EmaParquet=lpd.EmaGrid(temperature=20000.0,log_gravity=3.5,metalicity=0.0,filename="lp0000_20000_0350_0020.parquet"),Nadya=None)
grid2=lpd.IntensityGrid(EmaParquet=lpd.EmaGrid(temperature=20000.0,log_gravity=3.8,metalicity=0.0,filename="lp0000_20000_0380_0020.parquet"),Nadya=None)
grid3=lpd.IntensityGrid(EmaParquet=lpd.EmaGrid(temperature=24000.0,log_gravity=3.5,metalicity=0.0,filename="lp0000_24000_0350_0020.parquet"),Nadya=None)
grid4=lpd.IntensityGrid(EmaParquet=lpd.EmaGrid(temperature=24000.0,log_gravity=3.8,metalicity=0.0,filename="lp0000_24000_0380_0020.parquet"),Nadya=None)
prof_config=lpd.ProfileConfig(max_velocity=1.0e2,path_to_grids=path_to_grids,wavelength_range=wl_range,intensity_grids=[grid1,grid2,grid3,grid4])

prof_config_dict=prof_config.model_dump(exclude_none=True)

prof_toml_string=tomli_w.dumps(prof_config_dict)
print(prof_toml_string)

max_velocity = 100.0
path_to_grids = "/home/ricardo/Software/Pulstar/pulstarRust/profile/grids/ema_parquets/"

[wavelength_range]
start = 4551.0
end = 4555.0
step = 0.0033

[[intensity_grids]]

[intensity_grids.EmaParquet]
temperature = 20000.0
log_gravity = 3.5
metalicity = 0.0
filename = "lp0000_20000_0350_0020.parquet"

[[intensity_grids]]

[intensity_grids.EmaParquet]
temperature = 20000.0
log_gravity = 3.8
metalicity = 0.0
filename = "lp0000_20000_0380_0020.parquet"

[[intensity_grids]]

[intensity_grids.EmaParquet]
temperature = 24000.0
log_gravity = 3.5
metalicity = 0.0
filename = "lp0000_24000_0350_0020.parquet"

[[intensity_grids]]

[intensity_grids.EmaParquet]
temperature = 24000.0
log_gravity = 3.8
metalicity = 0.0
filename = "lp0000_24000_0380_0020.parquet"



# First run

In [10]:
pulse_df = psp.pulstar(puls_toml_string)

--------------------
--------------------
--------------------
|PULSTARust launched|
--------------------

 +-- Computing surface data for time point number 0 with time stamp 0.000.
----------------------
|PULSTARust Finished |
----------------------


In [15]:
pulse_df.sort("area").tail(5)

coord1,coord2,time,velocity,temperature,log gravity,coschi,area
f64,f64,f64,f64,f64,f64,f64,f64
0.94797,0.0,0.0,2.8544e-19,21671.810424,3.759759,0.986903,0.016123
0.730574,6.170986,0.0,-1.44352,21677.760057,3.75966,0.995554,0.016262
0.730574,0.1122,0.0,1.44352,21668.386268,3.76013,0.995554,0.016263
0.841069,0.098175,0.0,1.373394,21669.011085,3.760105,0.995931,0.016269
0.841069,6.185011,0.0,-1.373394,21677.215647,3.759693,0.995931,0.01627


In [16]:
wavelength_df = psp.profile(prof_toml_string,pulse_df)

----------------------------------------
----------------------------------------
[0.0]
min relative dopplershift is 0.9999524075484625
max relative dopplershift is 1.0000475924515375
creating the spectral grids data structures from csv files
Allocating memory for hypercube in the parameter space
Done
done computing flux
finished collecting fluxes 0
this is the df for mode 0: shape: (5, 5)
┌────────────┬──────────┬──────┬──────────┬───────────┐
│ wavelength ┆ pixel_id ┆ time ┆ flux     ┆ continuum │
│ ---        ┆ ---      ┆ ---  ┆ ---      ┆ ---       │
│ f64        ┆ u32      ┆ f64  ┆ f64      ┆ f64       │
╞════════════╪══════════╪══════╪══════════╪═══════════╡
│ 4551.0     ┆ 0        ┆ 0.0  ┆ 0.00399  ┆ 0.00399   │
│ 4551.0033  ┆ 1        ┆ 0.0  ┆ 0.003989 ┆ 0.00399   │
│ 4551.0066  �� 2        ┆ 0.0  ┆ 0.003989 ┆ 0.00399   │
│ 4551.0099  ┆ 3        ┆ 0.0  ┆ 0.003989 ┆ 0.00399   │
│ 4551.0132  ┆ 4        ┆ 0.0  ┆ 0.003989 ┆ 0.00399   │
└────────────┴──────────┴──────┴──────────┴───

In [17]:
wavelength_df.head(5)

wavelength,pixel_id,time,flux,continuum,normalized flux
f64,u32,f64,f64,f64,f64
4551.0,0,0.0,0.00399,0.00399,0.999787
4551.0033,1,0.0,0.003989,0.00399,0.999786
4551.0066,2,0.0,0.003989,0.00399,0.999785
4551.0099,3,0.0,0.003989,0.00399,0.999785
4551.0132,4,0.0,0.003989,0.00399,0.999784
